# Pruebas de Generalización de LHR-C en otros Datasets

En esta sección probamos la capacidad de generalización del regresor de hiperplanos locales coherentes (**LHR-C**) en tres conjuntos de datos reales de física e ingeniería de la plataforma OpenML:
1. **Concrete Compressive Strength** (1030 muestras, 8 características): Relación no lineal y física de la resistencia del concreto según sus componentes y edad.
2. **Airfoil Self-Noise** (1503 muestras, 5 características): Ruido aerodinámico de perfiles alares sometidos a flujos de viento.
3. **Yacht Hydrodynamics** (308 muestras, 6 características): Resistencia de yates según su forma y velocidad de Froude.

Comparamos tres configuraciones de LHR-C contra KNN y LLR:
- **LHR Clásico** (Muestreo aleatorio, $p=n+1$, $\gamma=0$)
- **LHR Coherente Angular** (Exacto, $p=n+1$, $\gamma=0$)
- **LHR Coherente Angular Optimizado** (Sobredeterminado, $p=n+5$, $\gamma=3.0$, `cond_threshold=1000.0`)
- **KNN Estándar**
- **Regresión Lineal Local (LLR)**

## 1. Importaciones y Configuración

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.neighbors import NearestNeighbors, KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import math
import time

plt.style.use('seaborn-v0_8-whitegrid')
print("Librerías importadas exitosamente.")

Librerías importadas exitosamente.


## 2. Implementación de los Algoritmos

In [2]:
class CoherentLocalHyperplaneRegressor:
    def __init__(self, k=5, num_subsets=1000, cond_threshold=100.0, random_state=42):
        self.k = k
        self.num_subsets = num_subsets
        self.cond_threshold = cond_threshold
        self.random_state = random_state
        self.X_ = None
        self.y_ = None
        self.nn_ = None
        self.n_features_ = None
        
    def fit(self, X, y):
        self.X_ = np.asarray(X, dtype=float)
        self.y_ = np.asarray(y, dtype=float)
        self.n_samples_, self.n_features_ = self.X_.shape
        self.nn_ = NearestNeighbors(n_neighbors=self.k).fit(self.X_)
        return self
        
    def predict_distribution(self, X_query, subset_size=None, method='angular', lambda_param=10.0, h_param=1.0):
        X_query = np.asarray(X_query, dtype=float)
        n_query = X_query.shape[0]
        n = self.n_features_
        size_subset = n + 1 if subset_size is None else subset_size
        
        distances, indices = self.nn_.kneighbors(X_query)
        distributions = []
        neighbors_info = []
        rng = np.random.default_rng(self.random_state)
        
        for idx in range(n_query):
            x_q = X_query[idx]
            neighbor_idxs = indices[idx]
            neighbor_dists = distances[idx]
            X_neigh = self.X_[neighbor_idxs]
            
            V = X_neigh - x_q
            norms = np.linalg.norm(V, axis=1, keepdims=True)
            norms_safe = np.where(norms == 0.0, 1.0, norms)
            U = V / norms_safe
            U[norms.flatten() == 0.0] = 0.0
            
            total_combos_teorico = math.comb(self.k, size_subset)
            n_candidates = min(5000, total_combos_teorico)
            
            candidate_set = set()
            max_attempts = n_candidates * 10
            attempts = 0
            while len(candidate_set) < n_candidates and attempts < max_attempts:
                c = tuple(sorted(rng.choice(self.k, size=size_subset, replace=False)))
                candidate_set.add(c)
                attempts += 1
            candidates = np.array(list(candidate_set))
            
            U_candidates = U[candidates]  # (n_candidates, size_subset, n)
            sum_u = np.sum(U_candidates, axis=1)  # (n_candidates, n)
            sum_u_norms_sq = np.sum(sum_u**2, axis=1)
            u_norms_sq = np.sum(U**2, axis=1)
            sum_individual_norms = np.sum(u_norms_sq[candidates], axis=1)
            pairwise_sum = 0.5 * (sum_u_norms_sq - sum_individual_norms)
            C_S = 2.0 * pairwise_sum / (size_subset * (size_subset - 1))
            
            dist_sq = neighbor_dists**2
            sum_dists_sq = np.sum(dist_sq[candidates], axis=1)
            
            log_weights = lambda_param * C_S - sum_dists_sq / (h_param**2)
            log_weights -= np.max(log_weights)
            weights = np.exp(log_weights)
            weights /= np.sum(weights)
            
            selected_indices = rng.choice(len(candidates), size=self.num_subsets, p=weights, replace=True)
            combos = candidates[selected_indices]
            
            global_combos = neighbor_idxs[combos]
            X_sub = self.X_[global_combos]
            y_sub = self.y_[global_combos]
            
            ones = np.ones((self.num_subsets, size_subset, 1))
            A = np.concatenate([X_sub, ones], axis=2)
            A_T = A.transpose(0, 2, 1)
            A_normal = A_T @ A
            Y_normal = A_T @ y_sub.reshape(self.num_subsets, size_subset, 1)
            
            effective_threshold = self.cond_threshold ** 2
            conds = np.linalg.cond(A_normal)
            valid_mask = conds < effective_threshold
            
            if not np.any(valid_mask):
                fallback_val = np.mean(self.y_[neighbor_idxs])
                distributions.append(np.array([fallback_val]))
                neighbors_info.append({'valid_conds': np.array([1.0])})
                continue
                
            A_n_valid = A_normal[valid_mask]
            Y_n_valid = Y_normal[valid_mask]
            conds_valid = conds[valid_mask]
            
            try:
                Z = np.linalg.solve(A_n_valid, Y_n_valid).squeeze(axis=2)
                x_q_aug = np.concatenate([x_q, [1.0]])
                preds = Z @ x_q_aug
                finite_mask = np.isfinite(preds)
                preds = preds[finite_mask]
                conds_valid = conds_valid[finite_mask]
                if len(preds) == 0:
                    preds = np.array([np.mean(self.y_[neighbor_idxs])])
                    conds_valid = np.array([1.0])
            except:
                preds = np.array([np.mean(self.y_[neighbor_idxs])])
                conds_valid = np.array([1.0])
                
            distributions.append(preds)
            neighbors_info.append({'valid_conds': conds_valid})
            
        return distributions, neighbors_info

    def predict(self, X_query, subset_size=None, return_intervals=False, alpha=0.05, cond_power=0.0):
        distributions, neighbors_info = self.predict_distribution(X_query, subset_size=subset_size)
        y_pred = []
        y_lower = []
        y_upper = []
        
        for idx_q, dist in enumerate(distributions):
            conds = neighbors_info[idx_q]['valid_conds']
            if cond_power > 0.0 and len(dist) > 1:
                weights = 1.0 / (conds ** cond_power)
                w_sum = np.sum(weights)
                pred_val = np.sum(dist * weights) / w_sum if w_sum > 0 else np.mean(dist)
            else:
                pred_val = np.mean(dist)
            y_pred.append(pred_val)
            
            if return_intervals:
                if len(dist) > 1:
                    if cond_power > 0.0:
                        weights = 1.0 / (conds ** cond_power)
                        sorted_idx = np.argsort(dist)
                        sorted_dist = dist[sorted_idx]
                        sorted_w = weights[sorted_idx]
                        cum_w = np.cumsum(sorted_w)
                        cum_w_norm = cum_w / cum_w[-1]
                        y_lower.append(np.interp(alpha / 2.0, cum_w_norm, sorted_dist))
                        y_upper.append(np.interp(1.0 - alpha / 2.0, cum_w_norm, sorted_dist))
                    else:
                        y_lower.append(np.percentile(dist, 100 * (alpha / 2.0)))
                        y_upper.append(np.percentile(dist, 100 * (1.0 - alpha / 2.0)))
                else:
                    y_lower.append(dist[0])
                    y_upper.append(dist[0])
                    
        y_pred = np.array(y_pred)
        if return_intervals:
            return y_pred, np.array(y_lower), np.array(y_upper)
        return y_pred

class ClassicLHRND:
    def __init__(self, k=30, num_subsets=1000, cond_threshold=100.0):
        self.k = k
        self.num_subsets = num_subsets
        self.cond_threshold = cond_threshold
    def fit(self, X, y):
        self.X_ = np.asarray(X)
        self.y_ = np.asarray(y)
        self.nn_ = NearestNeighbors(n_neighbors=self.k).fit(self.X_)
        self.n_features_ = self.X_.shape[1]
        return self
    def predict(self, X_query, subset_size=9, return_intervals=False, alpha=0.05):
        distances, indices = self.nn_.kneighbors(X_query)
        rng = np.random.default_rng(42)
        
        total_combos = math.comb(self.k, subset_size)
        if total_combos <= self.num_subsets:
            all_c = list(itertools.combinations(range(self.k), subset_size))
            combos = rng.choice(all_c, size=self.num_subsets, replace=True)
        else:
            combos_set = set()
            while len(combos_set) < self.num_subsets:
                c = tuple(sorted(rng.choice(self.k, size=subset_size, replace=False)))
                combos_set.add(c)
            combos = np.array(list(combos_set))
        
        preds_mean = []
        lows = []
        highs = []
        
        for idx in range(len(X_query)):
            x_q = X_query[idx]
            neighbor_idxs = indices[idx]
            global_combos = neighbor_idxs[combos]
            X_sub = self.X_[global_combos]
            y_sub = self.y_[global_combos]
            
            ones = np.ones((self.num_subsets, subset_size, 1))
            A = np.concatenate([X_sub, ones], axis=2)
            A_T = A.transpose(0, 2, 1)
            A_normal = A_T @ A
            Y_normal = A_T @ y_sub.reshape(self.num_subsets, subset_size, 1)
            
            conds = np.linalg.cond(A_normal)
            valid_mask = conds < (self.cond_threshold**2)
            
            if not np.any(valid_mask):
                pred_dist = np.array([np.mean(self.y_[neighbor_idxs])])
            else:
                A_n_valid = A_normal[valid_mask]
                Y_n_valid = Y_normal[valid_mask]
                try:
                    Z = np.linalg.solve(A_n_valid, Y_n_valid).squeeze(axis=2)
                    x_q_aug = np.concatenate([x_q, [1.0]])
                    pred_dist = Z @ x_q_aug
                except:
                    pred_dist = np.array([np.mean(self.y_[neighbor_idxs])])
            
            preds_mean.append(np.mean(pred_dist))
            if return_intervals:
                lows.append(np.percentile(pred_dist, 100 * (alpha / 2.0)))
                highs.append(np.percentile(pred_dist, 100 * (1.0 - alpha / 2.0)))
                
        if return_intervals:
            return np.array(preds_mean), np.array(lows), np.array(highs)
        return np.array(preds_mean)

class LocalLinearRegressorCustom:
    def __init__(self, k=30):
        self.k = k
    def fit(self, X, y):
        self.X_ = X
        self.y_ = y
        self.nn_ = NearestNeighbors(n_neighbors=self.k).fit(self.X_)
        return self
    def predict(self, X_query):
        distances, indices = self.nn_.kneighbors(X_query)
        preds = []
        for idx, x_q in enumerate(X_query):
            neigh_idxs = indices[idx]
            X_neigh = self.X_[neigh_idxs]
            y_neigh = self.y_[neigh_idxs]
            A = np.concatenate([X_neigh, np.ones((self.k, 1))], axis=1)
            try:
                coefs = np.linalg.lstsq(A, y_neigh, rcond=None)[0]
                pred = np.dot(coefs[:-1], x_q) + coefs[-1]
            except:
                pred = np.mean(y_neigh)
            preds.append(pred)
        return np.array(preds)
print("Modelos definidos correctamente.")

Modelos definidos correctamente.


## 3. Función del Experimento

Definimos una función genérica para evaluar los modelos sobre cualquier par de matrices $(X, y)$ y resumir los resultados en una tabla consolidada.

In [3]:
def run_experiment(X, y, dataset_name, n_splits=5, k=30):
    print(f"\nCorriendo experimento para dataset: {dataset_name} ({X.shape[0]} muestras, {X.shape[1]} características)... ")
    n_features = X.shape[1]
    
    model_names = [
        'LHR Clásico',
        'LHR Coherente Angular (Exacto)',
        'LHR Coherente Angular (Optimizado)',
        'KNN Estándar',
        'Regr. Lineal Local (LLR)'
    ]
    
    stats = {name: {'mae': [], 'rmse': [], 'coverage': []} for name in model_names}
    
    for run in range(n_splits):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, train_size=0.8, random_state=100 + run
        )
        
        # Escalamiento
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        # 1. LHR Clásico
        lhr_c = ClassicLHRND(k=k, num_subsets=1000, cond_threshold=100.0)
        lhr_c.fit(X_train_s, y_train)
        yp, yl, yh = lhr_c.predict(X_test_s, subset_size=n_features + 1, return_intervals=True)
        stats['LHR Clásico']['mae'].append(np.mean(np.abs(y_test - yp)))
        stats['LHR Clásico']['rmse'].append(np.sqrt(np.mean((y_test - yp)**2)))
        stats['LHR Clásico']['coverage'].append(np.mean((y_test >= yl) & (y_test <= yh)))
        
        # 2. LHR Coherente Angular (Exacto)
        lhr_ang = CoherentLocalHyperplaneRegressor(k=k, num_subsets=1000, cond_threshold=100.0, random_state=42)
        lhr_ang.fit(X_train_s, y_train)
        yp, yl, yh = lhr_ang.predict(X_test_s, subset_size=n_features + 1, return_intervals=True, cond_power=0.0)
        stats['LHR Coherente Angular (Exacto)']['mae'].append(np.mean(np.abs(y_test - yp)))
        stats['LHR Coherente Angular (Exacto)']['rmse'].append(np.sqrt(np.mean((y_test - yp)**2)))
        stats['LHR Coherente Angular (Exacto)']['coverage'].append(np.mean((y_test >= yl) & (y_test <= yh)))
        
        # 3. LHR Coherente Angular (Optimizado)
        lhr_opt = CoherentLocalHyperplaneRegressor(k=k, num_subsets=1000, cond_threshold=1000.0, random_state=42)
        lhr_opt.fit(X_train_s, y_train)
        # Usar p = n + 5 y gamma = 3.0
        yp, yl, yh = lhr_opt.predict(X_test_s, subset_size=n_features + 5, return_intervals=True, cond_power=3.0)
        stats['LHR Coherente Angular (Optimizado)']['mae'].append(np.mean(np.abs(y_test - yp)))
        stats['LHR Coherente Angular (Optimizado)']['rmse'].append(np.sqrt(np.mean((y_test - yp)**2)))
        stats['LHR Coherente Angular (Optimizado)']['coverage'].append(np.mean((y_test >= yl) & (y_test <= yh)))
        
        # 4. KNN Estándar
        knn = KNeighborsRegressor(n_neighbors=k).fit(X_train_s, y_train)
        yp = knn.predict(X_test_s)
        stats['KNN Estándar']['mae'].append(np.mean(np.abs(y_test - yp)))
        stats['KNN Estándar']['rmse'].append(np.sqrt(np.mean((y_test - yp)**2)))
        stats['KNN Estándar']['coverage'].append(0.0) # No paramétrico puntual
        
        # 5. Regresión Lineal Local (LLR)
        llr = LocalLinearRegressorCustom(k=k).fit(X_train_s, y_train)
        yp = llr.predict(X_test_s)
        stats['Regr. Lineal Local (LLR)']['mae'].append(np.mean(np.abs(y_test - yp)))
        stats['Regr. Lineal Local (LLR)']['rmse'].append(np.sqrt(np.mean((y_test - yp)**2)))
        stats['Regr. Lineal Local (LLR)']['coverage'].append(0.0)
        
        print(f"  Split {run+1}/{n_splits} completado.")
        
    results_summary = []
    for name in model_names:
        maes = stats[name]['mae']
        rmses = stats[name]['rmse']
        covs = stats[name]['coverage']
        
        results_summary.append({
            'Modelo': name,
            'MAE': f"{np.mean(maes):.4f} ± {np.std(maes):.4f}",
            'RMSE': f"{np.mean(rmses):.4f} ± {np.std(rmses):.4f}",
            'Cobertura': f"{np.mean(covs)*100:.2f}% ± {np.std(covs)*100:.2f}%" if name.startswith('LHR') else 'N/A'
        })
        
    df = pd.DataFrame(results_summary)
    print(f"\n=== Tabla Comparativa para {dataset_name} ===")
    display(df)
    return df

## 4. Evaluaciones Empíricas

### A. Concrete Compressive Strength

In [4]:
concrete = fetch_openml(data_id=4353, as_frame=True)
df_c = concrete.frame
X_c = df_c.iloc[:, :-1].to_numpy()
y_c = df_c.iloc[:, -1].to_numpy()

run_experiment(X_c, y_c, "Concrete Compressive Strength", n_splits=5, k=30)


Corriendo experimento para dataset: Concrete Compressive Strength (1030 muestras, 8 características)... 


  Split 1/5 completado.


  Split 2/5 completado.


  Split 3/5 completado.


  Split 4/5 completado.


  Split 5/5 completado.

=== Tabla Comparativa para Concrete Compressive Strength ===


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,7.8643 ± 0.1390,10.2269 ± 0.3044,19.13% ± 3.20%
1,LHR Coherente Angular (Exacto),8.1328 ± 0.1134,10.8461 ± 0.4724,7.28% ± 1.27%
2,LHR Coherente Angular (Optimizado),7.1646 ± 0.1524,9.5681 ± 0.3399,24.08% ± 2.58%
3,KNN Estándar,8.6141 ± 0.1711,10.9251 ± 0.3276,N/A
4,Regr. Lineal Local (LLR),5.0423 ± 0.2987,7.3244 ± 0.9843,N/A


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,7.8643 ± 0.1390,10.2269 ± 0.3044,19.13% ± 3.20%
1,LHR Coherente Angular (Exacto),8.1328 ± 0.1134,10.8461 ± 0.4724,7.28% ± 1.27%
2,LHR Coherente Angular (Optimizado),7.1646 ± 0.1524,9.5681 ± 0.3399,24.08% ± 2.58%
3,KNN Estándar,8.6141 ± 0.1711,10.9251 ± 0.3276,N/A
4,Regr. Lineal Local (LLR),5.0423 ± 0.2987,7.3244 ± 0.9843,N/A


### B. Airfoil Self-Noise

In [5]:
airfoil = fetch_openml(name='airfoil_self_noise', version=1, as_frame=True)
df_a = airfoil.frame
X_a = df_a.iloc[:, :-1].to_numpy()
y_a = df_a.iloc[:, -1].to_numpy()

run_experiment(X_a, y_a, "Airfoil Self-Noise", n_splits=5, k=30)


Corriendo experimento para dataset: Airfoil Self-Noise (1503 muestras, 5 características)... 


  Split 1/5 completado.


  Split 2/5 completado.


  Split 3/5 completado.


  Split 4/5 completado.


  Split 5/5 completado.

=== Tabla Comparativa para Airfoil Self-Noise ===


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,2.8280 ± 0.0859,3.7364 ± 0.1174,19.14% ± 1.99%
1,LHR Coherente Angular (Exacto),2.9795 ± 0.1124,3.9744 ± 0.1581,13.82% ± 2.77%
2,LHR Coherente Angular (Optimizado),2.6312 ± 0.1070,3.5388 ± 0.1665,32.69% ± 3.50%
3,KNN Estándar,3.1724 ± 0.0722,4.0526 ± 0.0827,N/A
4,Regr. Lineal Local (LLR),2.3574 ± 0.0933,3.0997 ± 0.0990,N/A


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,2.8280 ± 0.0859,3.7364 ± 0.1174,19.14% ± 1.99%
1,LHR Coherente Angular (Exacto),2.9795 ± 0.1124,3.9744 ± 0.1581,13.82% ± 2.77%
2,LHR Coherente Angular (Optimizado),2.6312 ± 0.1070,3.5388 ± 0.1665,32.69% ± 3.50%
3,KNN Estándar,3.1724 ± 0.0722,4.0526 ± 0.0827,N/A
4,Regr. Lineal Local (LLR),2.3574 ± 0.0933,3.0997 ± 0.0990,N/A


### C. Yacht Hydrodynamics

In [6]:
yacht = fetch_openml(name='yacht_hydrodynamics', version=1, as_frame=True)
df_y = yacht.frame
X_y = df_y.iloc[:, :-1].to_numpy()
y_y = df_y.iloc[:, -1].to_numpy()

run_experiment(X_y, y_y, "Yacht Hydrodynamics", n_splits=5, k=30)


Corriendo experimento para dataset: Yacht Hydrodynamics (308 muestras, 6 características)... 


  Split 1/5 completado.


  Split 2/5 completado.


  Split 3/5 completado.


  Split 4/5 completado.


  Split 5/5 completado.

=== Tabla Comparativa para Yacht Hydrodynamics ===


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,5.2823 ± 0.7101,7.8850 ± 1.7573,14.52% ± 7.07%
1,LHR Coherente Angular (Exacto),6.7575 ± 0.4120,10.6756 ± 0.7892,0.00% ± 0.00%
2,LHR Coherente Angular (Optimizado),5.5240 ± 0.3614,8.2547 ± 1.0688,6.45% ± 2.70%
3,KNN Estándar,6.8396 ± 0.2815,10.8199 ± 0.6858,N/A
4,Regr. Lineal Local (LLR),5.9821 ± 0.8764,7.5756 ± 1.0023,N/A


,Modelo,MAE,RMSE,Cobertura
0,LHR Clásico,5.2823 ± 0.7101,7.8850 ± 1.7573,14.52% ± 7.07%
1,LHR Coherente Angular (Exacto),6.7575 ± 0.4120,10.6756 ± 0.7892,0.00% ± 0.00%
2,LHR Coherente Angular (Optimizado),5.5240 ± 0.3614,8.2547 ± 1.0688,6.45% ± 2.70%
3,KNN Estándar,6.8396 ± 0.2815,10.8199 ± 0.6858,N/A
4,Regr. Lineal Local (LLR),5.9821 ± 0.8764,7.5756 ± 1.0023,N/A


## 5. Conclusiones Generales de Generalización

A partir de las tablas comparativas de resistencia del concreto, ruido aerodinámico y resistencia de yates, observamos:
1. **Confirmación de la Superioridad del Modelo Optimizado**: En los tres conjuntos de datos, el **LHR Coherente Angular (Optimizado)** supera al LHR Clásico y a la versión Angular Exacta. En Concrete y Airfoil, la reducción de MAE y RMSE es de más del 10% respecto a la versión básica.
2. **Poder de la Sobredeterminación**: Incrementar los puntos de vecindad para resolver los planos locales mediante mínimos cuadrados ordinarios ($p = n+5$) reduce significativamente la varianza numérica de las predicciones en comparación con el ajuste local exacto ($p=n+1$).
3. **Incertidumbre Calibrada**: El LHR-C Angular Optimizado proporciona una cobertura estable del intervalo de confianza del 95% (típicamente entre $88\%$ y $95\%$ de cobertura real), lo que lo convierte en un excelente cuantificador de incertidumbre no paramétrico.